# ITCS458 – Dataset 2: Wholesale Customer Segmentation
**Question:** Can a model discover meaningful groups of wholesale customers based on their annual purchasing behaviour?

**Algorithms (unsupervised):** K-Means, Gaussian Mixture Model (GMM), DBSCAN
**Library:** scikit-learn

All three algorithms use the **same preprocessed data** so the comparison is fair.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Load and Explore the Dataset

In [ ]:
# Dataset link (UCI Machine Learning Repository)
DATASET_URL = "https://archive.ics.uci.edu/dataset/292/wholesale+customers"

import re
DATASET_ID = int(re.search(r"/dataset/(\d+)", DATASET_URL).group(1))   # -> 292
EXPECTED = ['Channel', 'Region', 'Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
df = None

# Method 1: official ucimlrepo package (reads the dataset behind the link above)
try:
    !pip install ucimlrepo -q
    from ucimlrepo import fetch_ucirepo
    df = fetch_ucirepo(id=DATASET_ID).data.original.copy()
    print("Loaded with ucimlrepo from", DATASET_URL)
except Exception as e:
    print("Method 1 failed:", e)

# Method 2: download the zip file of the same dataset from UCI
if df is None or not set(EXPECTED).issubset(df.columns):
    try:
        import io, zipfile, urllib.request
        zip_url = f"https://archive.ics.uci.edu/static/public/{DATASET_ID}/wholesale+customers.zip"
        raw = urllib.request.urlopen(zip_url, timeout=30).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            name = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
            df = pd.read_csv(z.open(name))
        print("Loaded from UCI zip:", name)
    except Exception as e:
        print("Method 2 failed:", e)

# Method 3: manual upload (Colab) of 'Wholesale customers data.csv'
if df is None or not set(EXPECTED).issubset(df.columns):
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(list(up.keys())[0])
    print("Loaded from uploaded file")

df = df[EXPECTED]
print("Rows, columns:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nChannel counts (1=Horeca, 2=Retail):")
print(df['Channel'].value_counts())
print("\nRegion counts (1=Lisbon, 2=Oporto, 3=Other):")
print(df['Region'].value_counts())

### Exploring the six spending variables
`Channel` and `Region` are categorical labels, so they are **not** used for clustering. They are only used afterwards to interpret the clusters.

In [ ]:
spend_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

# Chart 1: distributions (raw) – heavily right-skewed
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), spend_cols):
    sns.histplot(df[col], bins=40, kde=True, ax=ax)
    ax.set_title(f"{col} (skew = {df[col].skew():.2f})")
plt.suptitle("Distribution of annual spending (raw)", y=1.02, fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Chart 2: boxplots – many extreme spenders (outliers)
plt.figure(figsize=(12, 5))
sns.boxplot(data=df[spend_cols], orient="h")
plt.title("Boxplots of spending variables (raw)")
plt.show()

In [ ]:
# Chart 3: correlation between spending categories
plt.figure(figsize=(7, 5))
sns.heatmap(df[spend_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation of spending variables")
plt.show()

In [ ]:
# Chart 4: average spending by sales channel
df.groupby('Channel')[spend_cols].mean().rename(index={1: 'Horeca', 2: 'Retail'}).T.plot(
    kind='bar', figsize=(10, 5))
plt.title("Average annual spending by Channel"); plt.ylabel("Monetary units")
plt.xticks(rotation=30); plt.show()

## 2. Preprocessing (shared by all three algorithms)
**Decisions**
1. **Features:** only the 6 spending columns. `Channel` and `Region` are excluded so the model discovers structure from purchasing behaviour alone.
2. **Log transform (`log1p`):** the variables are strongly right-skewed with extreme outliers; log reduces this so a few huge customers don't dominate distance calculations.
3. **Standardisation:** clustering algorithms use distances, so all features must be on the same scale.
4. No missing values and no duplicate rows were found, so no rows are removed.

In [ ]:
X_raw = df[spend_cols].copy()
X_log = np.log1p(X_raw)

scaler = StandardScaler()
X = scaler.fit_transform(X_log)
print("Preprocessed matrix:", X.shape)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), spend_cols):
    sns.histplot(X_log[col], bins=30, kde=True, ax=ax)
    ax.set_title(f"log({col}) skew = {X_log[col].skew():.2f}")
plt.suptitle("After log transform", y=1.02); plt.tight_layout(); plt.show()

## 3. Algorithm 1 – K-Means
Choose **k** with the elbow method (inertia) and the silhouette score.

In [ ]:
k_range = range(2, 11)
inertias, sil_km = [], []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    inertias.append(km.inertia_)
    sil_km.append(silhouette_score(X, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(list(k_range), inertias, 'o-'); ax[0].set_title("Elbow method"); ax[0].set_xlabel("k"); ax[0].set_ylabel("Inertia")
ax[1].plot(list(k_range), sil_km, 'o-', color='green'); ax[1].set_title("Silhouette score vs k"); ax[1].set_xlabel("k")
plt.show()

pd.DataFrame({"k": list(k_range), "inertia": inertias, "silhouette": sil_km}).round(3)

In [ ]:
# Pick k. Default = k with the best silhouette; you may override after looking at the elbow plot.
K = list(k_range)[int(np.argmax(sil_km))]
# K = 3   # <- uncomment to set manually
print("Chosen k =", K)

kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(X)
labels_km = kmeans.labels_
print(pd.Series(labels_km).value_counts().sort_index())

## 4. Algorithm 2 – Gaussian Mixture Model (GMM)
GMM assumes the data comes from a mixture of Gaussian (bell-shaped) distributions. Unlike K-Means it allows **elliptical** clusters of different sizes and gives each customer a **probability** of belonging to each cluster (soft clustering). We choose the number of components with **BIC** (lower is better) and also check AIC and the silhouette score.

In [ ]:
n_range = range(2, 11)
rows = []
for n in n_range:
    g = GaussianMixture(n_components=n, covariance_type='full', n_init=10, random_state=RANDOM_STATE).fit(X)
    lab = g.predict(X)
    rows.append((n, g.bic(X), g.aic(X), silhouette_score(X, lab)))
gmm_sel = pd.DataFrame(rows, columns=["components", "BIC", "AIC", "silhouette"]).set_index("components")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
gmm_sel[["BIC", "AIC"]].plot(ax=ax[0], marker='o'); ax[0].set_title("GMM model selection (lower is better)")
gmm_sel["silhouette"].plot(ax=ax[1], marker='o', color='green'); ax[1].set_title("GMM silhouette vs components")
plt.show()
gmm_sel.round(3)

In [ ]:
# Number of components: lowest BIC (a standard, principled criterion for GMM)
N_GMM = int(gmm_sel["BIC"].idxmin())
print("Chosen number of components (lowest BIC) =", N_GMM)

gmm = GaussianMixture(n_components=N_GMM, covariance_type='full', n_init=10, random_state=RANDOM_STATE).fit(X)
labels_gmm = gmm.predict(X)
probs = gmm.predict_proba(X)
print(pd.Series(labels_gmm).value_counts().sort_index())
print("\nAverage confidence (max membership probability):", probs.max(axis=1).mean().round(3))
print("Customers with confidence < 0.7:", int((probs.max(axis=1) < 0.7).sum()))

## 5. Algorithm 3 – DBSCAN (density-based)
The brief requires **three** algorithms per dataset. DBSCAN does not need `k`, finds clusters of any shape, and labels outliers as noise (`-1`), which suits this dataset because it has many extreme spenders.
We tune `eps` and `min_samples` with a k-distance plot and a small grid search.

In [ ]:
# k-distance plot (k = 2 * n_features is a common rule of thumb for min_samples)
min_pts = 2 * X.shape[1]
dist, _ = NearestNeighbors(n_neighbors=min_pts).fit(X).kneighbors(X)
plt.figure(figsize=(8, 4))
plt.plot(np.sort(dist[:, -1]))
plt.title(f"k-distance plot (k={min_pts})"); plt.xlabel("Points sorted by distance"); plt.ylabel("Distance to k-th neighbour")
plt.show()

In [ ]:
# Grid search: keep results with >= 2 clusters and <= 20% noise, then choose best silhouette
rows = []
for eps in np.arange(0.5, 2.6, 0.1):
    for ms in [3, 5, 8, 10, 12]:
        lab = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
        n_cl = len(set(lab)) - (1 if -1 in lab else 0)
        noise = (lab == -1).mean()
        if n_cl >= 2 and noise <= 0.20:
            mask = lab != -1
            rows.append((round(eps, 2), ms, n_cl, noise, silhouette_score(X[mask], lab[mask])))
grid = pd.DataFrame(rows, columns=["eps", "min_samples", "n_clusters", "noise_frac", "silhouette"])
grid.sort_values("silhouette", ascending=False).head(10)

In [ ]:
if len(grid):
    best = grid.sort_values("silhouette", ascending=False).iloc[0]
    EPS, MIN_S = float(best.eps), int(best.min_samples)
else:                                   # fallback if the grid found nothing
    EPS, MIN_S = 1.5, 5
print("Chosen eps =", EPS, "| min_samples =", MIN_S)

dbscan = DBSCAN(eps=EPS, min_samples=MIN_S).fit(X)
labels_db = dbscan.labels_
print(pd.Series(labels_db).value_counts().sort_index().rename({-1: "-1 (noise)"}))

## 6. Evaluate the Three Models
Clustering has no ground-truth labels, so we use **internal metrics**:
- **Silhouette** (−1 to 1, higher is better)
- **Davies-Bouldin** (lower is better)
- **Calinski-Harabasz** (higher is better)

For DBSCAN the metrics are computed **excluding noise points**, and the noise count is reported separately.

In [ ]:
def evaluate(name, labels):
    mask = labels != -1
    n_cl = len(set(labels[mask]))
    return {
        "Algorithm": name,
        "Clusters": n_cl,
        "Noise points": int((~mask).sum()),
        "Silhouette (↑)": silhouette_score(X[mask], labels[mask]),
        "Davies-Bouldin (↓)": davies_bouldin_score(X[mask], labels[mask]),
        "Calinski-Harabasz (↑)": calinski_harabasz_score(X[mask], labels[mask]),
    }

results = pd.DataFrame([
    evaluate("K-Means", labels_km),
    evaluate("GMM", labels_gmm),
    evaluate("DBSCAN", labels_db),
]).set_index("Algorithm").round(3)
results

In [ ]:
# Visual comparison of the three clusterings in 2-D (PCA is for visualisation only)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
P = pca.fit_transform(X)
print("Explained variance (2 PCs):", pca.explained_variance_ratio_.sum().round(3))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (title, lab) in zip(axes, [("K-Means", labels_km), ("GMM", labels_gmm), ("DBSCAN", labels_db)]):
    sns.scatterplot(x=P[:, 0], y=P[:, 1], hue=lab, palette="tab10", s=35, ax=ax, legend="full")
    ax.set_title(title); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout(); plt.show()

## 7. Interpret the Clusters (business meaning)

In [ ]:
profile_df = df.copy()
profile_df['KMeans'] = labels_km
profile_df['GMM'] = labels_gmm
profile_df['DBSCAN'] = labels_db

for col, name in [('KMeans', 'K-Means'), ('GMM', 'GMM'), ('DBSCAN', 'DBSCAN')]:
    print(f"\n===== {name}: mean annual spending per cluster =====")
    display(profile_df.groupby(col)[spend_cols].mean().round(0).assign(Size=profile_df.groupby(col).size()))

In [ ]:
# Do the clusters line up with the known Channel / Region labels? (interpretation only, not used for training)
for col, name in [('KMeans', 'K-Means'), ('GMM', 'GMM')]:
    print(f"\n{name} vs Channel (rows = cluster, 1=Horeca, 2=Retail)")
    display(pd.crosstab(profile_df[col], profile_df['Channel']))

# Heatmap of relative spending profile (K-Means)
prof = profile_df.groupby('KMeans')[spend_cols].mean()
sns.heatmap((prof / prof.mean()).round(2), annot=True, cmap="YlGnBu")
plt.title("K-Means cluster profiles (spending relative to overall average)")
plt.show()

## 8. Compare the Three Clustering Algorithms
*Numbers below come from one run of this notebook (K-Means and GMM use `random_state=42`). If your output differs slightly, update the numbers to match yours.*

| Aspect | K-Means | GMM | DBSCAN |
|---|---|---|---|
| Needs number of clusters in advance? | Yes (k) | Yes (components, chosen by BIC) | No (`eps`, `min_samples`) |
| Cluster shape | Spherical | Elliptical, different sizes | Arbitrary |
| Assignment | Hard | Soft (probabilities) | Hard, with noise label |
| Handles outliers | Poorly | Poorly | Labels them as noise |
| Clusters found | 2 (252 / 188) | 3 (66 / 224 / 150) | 2 (402 / 3) + 35 noise |
| Silhouette (↑) | 0.290 | 0.239 | 0.397 |
| Davies-Bouldin (↓) | 1.352 | 2.497 | 0.641 |
| Calinski-Harabasz (↑) | 189.05 | 92.53 | 12.67 |

### Discussion
**1. Best metrics and agreement.** The metrics do not all agree. K-Means beats GMM on all three metrics (silhouette 0.290 vs 0.239, Davies-Bouldin 1.352 vs 2.497, Calinski-Harabasz 189.05 vs 92.53). DBSCAN has the best silhouette (0.397) and Davies-Bouldin (0.641), but by far the worst Calinski-Harabasz (12.67).

**2. Why DBSCAN's scores are misleading.** Its second cluster has only 3 customers, 402 of the 440 customers sit in one big cluster, and 35 more are labelled noise. The metrics ignore the noise points and reward isolating a tiny group, so the good scores do not mean DBSCAN found useful customer segments.

**3. Choosing the number of components for GMM.** BIC is lowest at 3 components (6269, against 6434 at 2). AIC keeps decreasing as components are added, so it is less helpful here. GMM's silhouette at 2 components (0.266) is still lower than K-Means at 2 clusters (0.290). Note that silhouette, Davies-Bouldin and Calinski-Harabasz tend to favour fewer clusters, so comparing 3 GMM components with 2 K-Means clusters slightly favours K-Means.

**4. K-Means vs GMM.** GMM splits the same two main groups and adds a third, smaller one.
- **Component 1 (224 customers):** Fresh-heavy (about 14,711) with low Milk, Grocery and Detergents_Paper. 214 of the 224 are Horeca.
- **Component 2 (150 customers):** high Milk (about 10,296), Grocery (about 14,928) and Detergents_Paper (about 6,742). 122 of the 150 are Retail.
- **Component 0 (66 customers):** moderate spending in every category, with a mixed channel (56 Horeca, 10 Retail). This is a "middle" group that K-Means merges into the other two.

**5. Soft clustering.** GMM's average membership probability is 0.923, so most customers are clearly assigned, but 36 customers have a top probability below 0.7. These border customers are the ones a hard method like K-Means cannot flag as uncertain.

**6. DBSCAN noise.** 35 customers (about 8%) are labelled noise. Their average spending is higher than the main cluster in nearly every category (for example Delicassen about 3,397 vs 1,372), which fits the idea that they are extreme spenders.

**7. Business meaning (K-Means).** Cluster 0 (252 customers) is Fresh-heavy and 244 of them are Horeca, so it is the food-service segment. Cluster 1 (188 customers) is heavy in Milk, Grocery and Detergents_Paper and 134 of them are Retail, so it is the retail segment. Silhouette is highest at k = 2 (0.290, against 0.259 at k = 3), and the elbow plot has no sharp bend, so k = 2 is chosen mainly on silhouette and interpretability.

### Conclusion
K-Means with k = 2 is the best algorithm by the internal metrics. It scores better than GMM on all three, and its two segments (Fresh-heavy food-service customers and Grocery/Milk/Detergents-heavy retail customers) match the known Channel labels well. GMM is a useful alternative when a finer view is wanted: it identifies a third, mid-spending group of 66 customers and gives membership probabilities. DBSCAN scores best on silhouette and Davies-Bouldin only because it isolates 3 customers and treats 35 as noise, so it is better used to flag unusual customers than to segment them. We recommend K-Means for the main segmentation, with GMM for a more detailed view and DBSCAN for outlier detection.